In [3]:
import requests
import zipfile
import io
import os
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [24]:
con = duckdb.connect("/workspaces/dezoomcamp-divvy-chicago-dbt/data/warehouse.duckdb")

In [21]:
con.close()

In [25]:
# con.execute("SHOW TABLES").df()

con.execute("""
SELECT *
FROM information_schema.tables
WHERE table_schema = 'main'
""").df()

,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action,TABLE_COMMENT
0,warehouse,main,dim_stations,BASE TABLE,None,None,None,None,None,YES,NO,None,None
1,warehouse,main,fct_trips,BASE TABLE,None,None,None,None,None,YES,NO,None,None
2,warehouse,main,my_first_dbt_model,BASE TABLE,None,None,None,None,None,YES,NO,None,None
3,warehouse,main,peak_usage,BASE TABLE,None,None,None,None,None,YES,NO,None,None
4,warehouse,main,trips_by_day,BASE TABLE,None,None,None,None,None,YES,NO,None,None
5,warehouse,main,trips_by_station,BASE TABLE,None,None,None,None,None,YES,NO,None,None
6,warehouse,main,my_second_dbt_model,VIEW,None,None,None,None,None,NO,NO,None,None
7,warehouse,main,stg_divvy,VIEW,None,None,None,None,None,NO,NO,None,None
8,warehouse,main,stg_divvy_trips,VIEW,None,None,None,None,None,NO,NO,None,None


In [ ]:
df = con.execute("""
    SELECT *
    FROM read_parquet('/workspaces/dezoomcamp-divvy-chicago-dbt/data/raw/*.parquet')
    LIMIT 10
""").df()

df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,C1D650626C8C899A,electric_bike,2024-01-12 15:30:27,2024-01-12 15:37:59,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.903267,-87.634737,41.889177,-87.638506,member
1,EECD38BDB25BFCB0,electric_bike,2024-01-08 15:45:46,2024-01-08 15:52:59,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902937,-87.634440,41.889177,-87.638506,member
2,F4A9CE78061F17F7,electric_bike,2024-01-27 12:27:19,2024-01-27 12:35:19,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902951,-87.634470,41.889177,-87.638506,member
3,0A0D9E15EE50B171,classic_bike,2024-01-29 16:26:17,2024-01-29 16:56:06,Wells St & Randolph St,TA1305000030,Larrabee St & Webster Ave,13193,41.884295,-87.633963,41.921822,-87.644140,member
4,33FFC9805E3EFF9A,classic_bike,2024-01-31 05:43:23,2024-01-31 06:09:35,Lincoln Ave & Waveland Ave,13253,Kingsbury St & Kinzie St,KA1503000043,41.948797,-87.675278,41.889177,-87.638506,member


In [27]:
con.execute("""
SELECT *
FROM fct_trips
LIMIT 10
""").df()

,ride_id,started_at,ended_at,trip_date,trip_hour,trip_duration_min,start_station_name,end_station_name,member_casual
0,0000033BAC7DD047,2024-09-02 11:52:27.631,2024-09-02 12:10:31.618,2024-09-02,11,18,None,None,member
1,00006CC95E4CCEF4,2024-09-01 19:11:41.396,2024-09-01 19:17:32.098,2024-09-01,19,6,Racine Ave & Fullerton Ave,None,member
2,0000B9E8F22A2C2F,2024-12-22 15:47:26.259,2024-12-22 15:54:06.927,2024-12-22,15,7,Wells St & Walton St,Halsted St & Clybourn Ave,member
3,0000BB813C8D3183,2024-04-13 19:10:55.000,2024-04-13 19:18:54.000,2024-04-13,19,8,Montrose Harbor,Montrose Harbor,casual
4,0000D0E369788F4D,2024-06-22 11:46:18.276,2024-06-22 12:21:26.642,2024-06-22,11,35,Streeter Dr & Grand Ave,Ritchie Ct & Banks St,casual
5,0000DE1BB9C03273,2024-05-21 07:56:08.000,2024-05-21 08:07:33.000,2024-05-21,7,11,None,None,member
6,0000E29F50F444CF,2024-06-11 11:05:04.916,2024-06-11 11:11:13.508,2024-06-11,11,6,Green St & Washington Blvd,Elizabeth St & Fulton St,casual
7,000188E8C1963FD7,2024-10-04 16:42:08.597,2024-10-04 16:59:11.902,2024-10-04,16,17,Clinton St & Washington Blvd,Smith Park,member
8,0001941A2BC4AFB7,2024-01-25 17:29:06.000,2024-01-25 18:05:03.000,2024-01-25,17,36,DuSable Lake Shore Dr & Monroe St,Lake Park Ave & 47th St,member
9,0001BD2026812971,2024-08-23 18:10:58.020,2024-08-23 18:24:56.634,2024-08-23,18,14,None,None,member


In [ ]:
con.execute("""
  SELECT *
  FROM dim_stations
  LIMIT 10
  """).df()

,station_name,lat,lng
0,Wabash Ave & Roosevelt Rd,41.867227,-87.625961
1,Ada St & Washington Blvd,41.882830,-87.661206
2,Griffin Museum of Science and Industry,41.791728,-87.583945
3,Clarendon Ave & Junior Ter,41.961004,-87.649603
4,Ashland Ave & Blackhawk St,41.907066,-87.667252
5,DuSable Lake Shore Dr & Belmont Ave,41.940775,-87.639192
6,Milwaukee Ave & Wabansia Ave,41.912589,-87.681386
7,Ashland Ave & Wellington Ave,41.936083,-87.669807
8,Theater on the Lake,41.926277,-87.630834
9,Wabash Ave & Grand Ave,41.891466,-87.626761


In [ ]:
con.execute(""" 
  SELECT * 
  FROM peak_usage
  LIMIT 10""").df()

,trip_hour,trips
0,0,66643
1,1,42788
2,2,25544
3,3,15661
4,4,14915


In [31]:
con.execute(""" 
  SELECT * 
  FROM trips_by_day
  LIMIT 10""").df()

,trip_date,total_trips,avg_duration
0,2024-01-01,3604,20.888457
1,2024-01-02,6435,10.705983
2,2024-01-03,7366,11.910399
3,2024-01-04,8004,12.064093
4,2024-01-05,7319,11.826889
5,2024-01-06,3504,14.933505
6,2024-01-07,4365,11.115464
7,2024-01-08,7936,12.710685
8,2024-01-09,3222,12.533209
9,2024-01-10,7752,10.997936


In [32]:
con.execute(""" 
  SELECT * 
  FROM trips_by_station
  LIMIT 10""").df()

,start_station_name,trip_count
0,None,1031714
1,Streeter Dr & Grand Ave,65527
2,DuSable Lake Shore Dr & Monroe St,43610
3,Kingsbury St & Kinzie St,39429
4,Michigan Ave & Oak St,39312
5,DuSable Lake Shore Dr & North Blvd,39241
6,Clark St & Elm St,35295
7,Clinton St & Washington Blvd,34310
8,Millennium Park,32911
9,Clinton St & Madison St,32800
